# Detecting Fraudulent Bank Account Applications under Extreme Class Imbalance
### A Systematic Comparison of Six Classifiers — *Project Phase 2*

**Nourhan Hamdy** · Computer Engineering Department · Arab Academy for Science, Technology and Maritime Transport · Alexandria, Egypt

---
This notebook reproduces the full machine-learning pipeline end to end:

1. **Data preprocessing** — sentinel decoding, missing-indicator flags, imputation, encoding, scaling
2. **Model implementation** — 6 classifiers + a stacking ensemble on one shared split
3. **Ablation study** — per-model grid search with stratified *k*-fold cross-validation (PR-AUC)
4. **Evaluation & comparison** — held-out metrics, ROC / PR curves, confusion matrices
5. **Interpretation** — why simple additive models win, and the SMOTE leakage trap

**Dataset:** Bank Account Fraud (BAF) *Base* — 1,000,000 applications, target `fraud_bool`, **1.10 % fraud (~90:1 imbalance)**.

**Reproducibility:** every random operation is seeded with `SEED = 42`.

## 0. Setup & configuration

In [ ]:
import json, time, pickle, warnings, gc
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

# --- Path to the dataset: edit if needed ---
DATA_PATH = "Base.csv"          # e.g. "/path/to/Base.csv"

TARGET = "fraud_bool"
SAMPLE_N = 50_000               # stratified subsample for SVM/KNN tractability
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
print("scikit-learn", sklearn.__version__)

## 1. Data Preprocessing

### 1.1 Load and inspect
We first profile the full file. Key facts discovered during EDA:
- **No explicit `NaN`**, but `-1` is an *implicit missingness sentinel* in several count features.
- `prev_address_months_count` is **71 %** sentinel and `bank_months_count` **25 %**.
- `device_fraud_count` is **constant (all zeros)** → dropped (zero variance).
- Severe imbalance: only **1.10 %** of rows are fraud.

In [ ]:
df_full = pd.read_csv(DATA_PATH)
print("Full shape:", df_full.shape)
print("Fraud rate: {:.4f}  ({} fraud / {} total)".format(
      df_full[TARGET].mean(), int(df_full[TARGET].sum()), len(df_full)))
print("Imbalance ratio: {:.1f} : 1".format((df_full[TARGET]==0).sum()/(df_full[TARGET]==1).sum()))
df_full.head(3)

In [ ]:
# Sentinel (-1) missingness rates and the constant column
num_cols_all = df_full.select_dtypes(include=[np.number]).columns.drop(TARGET)
sentinel_rates = {c: float((df_full[c] == -1).mean()) for c in num_cols_all
                  if (df_full[c] == -1).mean() > 0.0001}
print("Sentinel (-1) rates:")
for k, v in sorted(sentinel_rates.items(), key=lambda x: -x[1]):
    print(f"  {k:35s} {v:6.1%}")
print("\nConstant columns:", [c for c in df_full.columns if df_full[c].nunique() <= 1])

### 1.2 Stratified subsample
The RBF SVM and *k*-NN scale super-linearly in sample size, so we draw a single **stratified** 
50,000-row subsample that **preserves the exact fraud rate**, then free the full frame from memory.
*(Remove this step to run on the full dataset if compute allows — it is the main study limitation.)*

In [ ]:
frac = SAMPLE_N / len(df_full)
parts = [g.sample(int(round(len(g) * frac)), random_state=SEED)
         for _, g in df_full.groupby(TARGET)]
df = pd.concat(parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
del df_full, parts; gc.collect()
print("Subsample:", df.shape, "| fraud rate {:.4f}".format(df[TARGET].mean()))

### 1.3 Feature groups & missing-indicator strategy
We decode `-1` → `NaN`, then add **binary missing-indicator flags** for the two high-missingness 
features. The act of missingness is itself informative — in fact the *missing-previous-address* flag 
turns out to be the single strongest predictor of fraud.

| Group | Treatment | Rationale |
|-------|-----------|-----------|
| Numeric | median impute → standardize | needed by distance/linear models |
| Categorical (≤7 levels) | one-hot | nominal, low cardinality |
| Binary 0/1 flags | passthrough | already encoded |
| `device_fraud_count` | **drop** | zero variance |

In [ ]:
df = df.drop(columns=["device_fraud_count"])   # constant

CAT_COLS = ["payment_type", "employment_status", "housing_status", "source", "device_os"]
BIN_COLS = ["email_is_free", "phone_home_valid", "phone_mobile_valid",
            "has_other_cards", "foreign_request", "keep_alive_session"]
SENTINEL = ["prev_address_months_count", "current_address_months_count", "bank_months_count",
            "session_length_in_minutes", "device_distinct_emails_8w", "credit_risk_score"]

y = df[TARGET].values
X = df.drop(columns=[TARGET]).copy()
NUM_COLS = [c for c in X.columns if c not in CAT_COLS + BIN_COLS]

# -1 sentinel -> NaN, plus missing-indicator flags for the two high-missingness features
for c in SENTINEL:
    X.loc[X[c] == -1, c] = np.nan
X["flag_prev_addr_missing"]  = X["prev_address_months_count"].isna().astype(int)
X["flag_bank_months_missing"] = X["bank_months_count"].isna().astype(int)
BIN_COLS = BIN_COLS + ["flag_prev_addr_missing", "flag_bank_months_missing"]

print(f"numeric={len(NUM_COLS)}  categorical={len(CAT_COLS)}  binary={len(BIN_COLS)}")

### 1.4 Train / test split + preprocessing pipeline
A single **stratified 80/20 split** is shared by every model. The `ColumnTransformer` is **fit on the 
training split only** and applied identically to the test split — no leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)

num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale",  StandardScaler())])
cat_pipe = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

pre = ColumnTransformer([
    ("num", num_pipe, NUM_COLS),
    ("cat", cat_pipe, CAT_COLS),
    ("bin", "passthrough", BIN_COLS),
], remainder="drop")

pre.fit(X_tr)
Xtr = pre.transform(X_tr).astype(np.float32)
Xte = pre.transform(X_te).astype(np.float32)
feat_names = (NUM_COLS
              + list(pre.named_transformers_["cat"]["onehot"].get_feature_names_out(CAT_COLS))
              + BIN_COLS)

print("train:", Xtr.shape, "| test:", Xte.shape, "| features:", len(feat_names))
print("train fraud:", int(y_tr.sum()), "| test fraud:", int(y_te.sum()))

### 1.5 Class-imbalance treatment
Primary strategy is **cost-sensitive class weighting** (`class_weight='balanced'`) wherever supported. 
For *k*-NN, which has no native weighting, we instead apply **SMOTE** to the training split. 
We keep SMOTE deliberately to demonstrate its validation-leakage pitfall later.

> **Note:** SMOTE here is applied *before* cross-validation purely to expose the leakage effect. 
> In production it must live *inside* the CV loop (e.g. via `imblearn.pipeline.Pipeline`).

In [ ]:
from sklearn.neighbors import NearestNeighbors

def smote(Xa, ya, k=5, seed=SEED):
    """Minimal SMOTE: interpolate between minority-class nearest neighbours."""
    rng = np.random.RandomState(seed)
    Xa = np.asarray(Xa, np.float32); ya = np.asarray(ya)
    n_maj, n_min = (ya == 0).sum(), (ya == 1).sum()
    need = n_maj - n_min
    Xmin = Xa[ya == 1]
    _, idx = NearestNeighbors(n_neighbors=k + 1).fit(Xmin).kneighbors(Xmin)
    syn = np.empty((need, Xa.shape[1]), np.float32)
    for i in range(need):
        a = rng.randint(len(Xmin)); nb = idx[a, rng.randint(1, k + 1)]
        syn[i] = Xmin[a] + rng.rand() * (Xmin[nb] - Xmin[a])
    Xr = np.vstack([Xa, syn]); yr = np.concatenate([ya, np.ones(need, int)])
    p = rng.permutation(len(yr))
    return Xr[p], yr[p]

Xtr_sm, ytr_sm = smote(Xtr, y_tr)
print("Before SMOTE:", dict(zip(*np.unique(y_tr, return_counts=True))))
print("After  SMOTE:", dict(zip(*np.unique(ytr_sm, return_counts=True))))

### 1.6 Exploratory figures
Two takeaways: (a) the extreme imbalance makes accuracy meaningless, and (b) **every** feature's 
correlation with the target is `|r| < 0.1` — there is no single separating feature, which favours 
additive models that combine many weak cues.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
vc = pd.Series(y_tr).value_counts().sort_index()
ax[0].bar(["Legit (0)", "Fraud (1)"], vc.values, color=["#4C72B0", "#C44E52"])
ax[0].set_title("Class distribution (train)"); ax[0].set_ylabel("count")
for i, v in enumerate(vc.values):
    ax[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)

corr = pd.DataFrame(Xtr, columns=feat_names).assign(y=y_tr).corr()["y"].drop("y")
top = corr.reindex(corr.abs().sort_values(ascending=False).index).head(12)
ax[1].barh(top.index[::-1], top.values[::-1],
           color=["#C44E52" if v > 0 else "#4C72B0" for v in top.values[::-1]])
ax[1].set_title("Top 12 |correlation| with fraud"); ax[1].set_xlabel("Pearson r")
ax[1].tick_params(labelsize=7)
plt.tight_layout(); plt.show()

## 2 & 3. Model Implementation + Ablation Study

For every model we define a search space of ≥3 configurations and run an exhaustive **grid search** 
with **stratified *k*-fold cross-validation** on the training split only, scored by **average precision 
(PR-AUC)** — the appropriate metric under extreme imbalance. The winning configuration is refit on 
the full training split and frozen before the test set is ever touched.

All models share the same split and the fixed seed. We record **training time** for each.

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier,
                              StackingClassifier)

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = "average_precision"          # PR-AUC
ablation, best_est, train_time, cv_score = {}, {}, {}, {}

def run_grid(name, estimator, grid, Xfit, yfit, cv=cv5):
    t0 = time.time()
    gs = GridSearchCV(estimator, grid, scoring=SCORING, cv=cv, n_jobs=1, refit=True)
    gs.fit(Xfit, yfit)
    dt = time.time() - t0
    res = pd.DataFrame(gs.cv_results_)
    keep = [c for c in res.columns if c.startswith("param_")] + \
           ["mean_test_score", "std_test_score", "rank_test_score"]
    ablation[name]   = res[keep].sort_values("rank_test_score").reset_index(drop=True)
    best_est[name]   = gs.best_estimator_
    train_time[name] = dt
    cv_score[name]   = gs.best_score_
    print(f"[{name:13s}] CV PR-AUC={gs.best_score_:.4f}  best={gs.best_params_}  ({dt:.1f}s)")
    return gs

### 2.1 Logistic Regression — search `C ∈ {0.01, 0.1, 1, 10}`

In [ ]:
run_grid("LogReg",
         LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
         {"C": [0.01, 0.1, 1.0, 10.0]}, Xtr, y_tr)
ablation["LogReg"]

### 2.2 Decision Tree — search `max_depth × min_samples_leaf`

In [ ]:
run_grid("DecisionTree",
         DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
         {"max_depth": [3, 5, 10, None], "min_samples_leaf": [1, 20]}, Xtr, y_tr)
ablation["DecisionTree"].head()

### 2.3 *k*-Nearest Neighbors — trained on SMOTE-balanced data
`algorithm='brute'` is far faster than `kd_tree` in this 53-dimensional space. 
Watch the cross-validation PR-AUC here — it will look *too* good (the leakage trap).

In [ ]:
run_grid("KNN",
         KNeighborsClassifier(algorithm="brute", n_jobs=1),
         {"n_neighbors": [11, 25, 51], "weights": ["uniform", "distance"]},
         Xtr_sm, ytr_sm)
ablation["KNN"]

### 2.4 SVM (RBF) — search `C × gamma`
The RBF SVM scales ~O(n²) and Platt probability calibration would add a 5× CV cost, so we **rank with 
`decision_function`** (monotone → exact ROC/PR) and refit the winner on a documented 20k subsample.

In [ ]:
rs = np.random.RandomState(SEED)
i0, i1 = np.where(y_tr == 0)[0], np.where(y_tr == 1)[0]
sub = np.concatenate([rs.choice(i0, min(len(i0), 11600), replace=False), i1]); rs.shuffle(sub)

gs_svm = run_grid("SVM",
         SVC(kernel="rbf", class_weight="balanced", random_state=SEED),
         {"C": [0.1, 1.0, 10.0], "gamma": ["scale", 0.1]},
         Xtr[sub], y_tr[sub],
         cv=StratifiedKFold(3, shuffle=True, random_state=SEED))

# refit best SVM on a 20k stratified subsample
r2 = np.random.RandomState(SEED + 1)
sub2 = np.concatenate([r2.choice(i0, min(len(i0), 19500), replace=False), i1]); r2.shuffle(sub2)
t0 = time.time()
svm_best = SVC(kernel="rbf", class_weight="balanced", random_state=SEED,
               **gs_svm.best_params_).fit(Xtr[sub2], y_tr[sub2])
train_time["SVM"] = time.time() - t0
best_est["SVM"] = svm_best
print(f"[SVM] refit on 20k subsample ({train_time['SVM']:.1f}s)")

### 2.5 Random Forest — search `max_depth × min_samples_leaf` (150 trees)

In [ ]:
run_grid("RandomForest",
         RandomForestClassifier(n_estimators=150, class_weight="balanced",
                                random_state=SEED, n_jobs=1),
         {"max_depth": [12, 20], "min_samples_leaf": [5, 20]}, Xtr, y_tr)
ablation["RandomForest"]

### 2.6 Histogram Gradient Boosting *(ensemble method)*
A modern histogram-based gradient booster (the scikit-learn equivalent of LightGBM/XGBoost), 
typically the strongest tabular baseline.

In [ ]:
run_grid("HistGB",
         HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED,
                                        early_stopping=True, validation_fraction=0.15),
         {"learning_rate": [0.05, 0.1], "max_depth": [None, 6], "max_iter": [300, 600]},
         Xtr, y_tr)
ablation["HistGB"].head()

### 2.7 Stacking ensemble *(second ensemble method)*
Combines the tuned Random Forest, Gradient Boosting and Logistic Regression base learners through a 
logistic meta-learner over out-of-fold predictions.

In [ ]:
t0 = time.time()
stack = StackingClassifier(
    estimators=[("rf",  best_est["RandomForest"]),
                ("hgb", best_est["HistGB"]),
                ("lr",  best_est["LogReg"])],
    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
    cv=3, stack_method="predict_proba", n_jobs=1)
stack.fit(Xtr, y_tr)
train_time["Stacking"] = time.time() - t0
best_est["Stacking"] = stack
cv_score["Stacking"] = np.nan
print(f"[Stacking] fit ({train_time['Stacking']:.1f}s)")

## 4. Evaluation & Comparison (held-out test set)

We score every frozen model on the **single shared test set** with a common metric suite. 
`Recall@5%FPR` is our **operational metric**: how much fraud we catch within a 5 % false-positive 
(analyst-review) budget. SVM is thresholded at its native decision boundary; others at 0.5.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, precision_recall_curve)

def scores_of(name, est):
    if name == "SVM":
        raw = est.decision_function(Xte)
        s = (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)
        yp = est.predict(Xte)
    else:
        s = est.predict_proba(Xte)[:, 1]
        yp = (s >= 0.5).astype(int)
    return s, yp

def tpr_at_fpr(y, s, tf=0.05):
    fpr, tpr, _ = roc_curve(y, s)
    return float(tpr[max(np.searchsorted(fpr, tf, side="right") - 1, 0)])

rows, roc_data, pr_data, cms = [], {}, {}, {}
for name, est in best_est.items():
    s, yp = scores_of(name, est)
    rows.append({"Model": name,
                 "Accuracy":  accuracy_score(y_te, yp),
                 "Precision": precision_score(y_te, yp, zero_division=0),
                 "Recall":    recall_score(y_te, yp, zero_division=0),
                 "F1":        f1_score(y_te, yp, zero_division=0),
                 "ROC_AUC":   roc_auc_score(y_te, s),
                 "PR_AUC":    average_precision_score(y_te, s),
                 "Recall@5%FPR": tpr_at_fpr(y_te, s),
                 "CV_PR_AUC": cv_score.get(name, np.nan),
                 "Train_s":   train_time.get(name, np.nan)})
    fpr, tpr, _ = roc_curve(y_te, s);            roc_data[name] = (fpr, tpr)
    prec, rec, _ = precision_recall_curve(y_te, s); pr_data[name] = (rec, prec)
    cms[name] = confusion_matrix(y_te, yp)

comparison = (pd.DataFrame(rows).sort_values("PR_AUC", ascending=False)
              .reset_index(drop=True).round(4))
comparison

### 4.1 ROC and Precision–Recall curves

In [ ]:
order = comparison["Model"].tolist()
cmap = dict(zip(order, plt.cm.tab10(np.linspace(0, 1, 10))))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))

for n in order:
    fpr, tpr = roc_data[n]
    ax[0].plot(fpr, tpr, lw=1.6, color=cmap[n],
               label=f"{n} ({comparison.set_index('Model').loc[n,'ROC_AUC']:.3f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].axvline(0.05, color="grey", ls=":", lw=.8)
ax[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC curves")
ax[0].legend(fontsize=7, loc="lower right")

for n in order:
    rec, prec = pr_data[n]
    ax[1].plot(rec, prec, lw=1.6, color=cmap[n],
               label=f"{n} ({comparison.set_index('Model').loc[n,'PR_AUC']:.3f})")
ax[1].axhline(y_te.mean(), color="grey", ls="--", lw=.8, label=f"baseline={y_te.mean():.3f}")
ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall curves")
ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

### 4.2 Confusion matrices @ threshold

In [ ]:
nm = len(order); cols = 4; rwsn = int(np.ceil(nm / cols))
fig, axes = plt.subplots(rwsn, cols, figsize=(3 * cols, 2.7 * rwsn))
for ax_, n in zip(axes.ravel(), order):
    cm = cms[n]; ax_.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax_.text(j, i, f"{v:,}", ha="center", va="center",
                 color="white" if v > cm.max() / 2 else "black", fontsize=8)
    ax_.set_title(n, fontsize=9)
    ax_.set_xticks([0, 1]); ax_.set_yticks([0, 1])
    ax_.set_xticklabels(["Legit", "Fraud"], fontsize=7)
    ax_.set_yticklabels(["Legit", "Fraud"], fontsize=7)
    ax_.set_xlabel("Predicted", fontsize=7); ax_.set_ylabel("Actual", fontsize=7)
for ax_ in axes.ravel()[nm:]: ax_.axis("off")
plt.suptitle("Confusion matrices (held-out test)", y=1.0)
plt.tight_layout(); plt.show()

## 5. Interpretation & Discussion

### 5.1 The SMOTE validation-leakage trap
The most instructive result is *k*-NN: a cross-validation PR-AUC near **0.99** that collapses to ~**0.05** 
on the real test set. Because SMOTE was applied to the whole training pool *before* cross-validation, 
synthetic minority points interpolated from training neighbours leaked into the validation folds — the 
model was effectively scored on near-copies of its own data. The lesson: **resampling must live inside 
the CV loop.**

In [ ]:
knn = comparison.set_index("Model").loc["KNN"]
print(f"k-NN cross-validation PR-AUC : {knn['CV_PR_AUC']:.3f}  (looked spectacular)")
print(f"k-NN real held-out  PR-AUC : {knn['PR_AUC']:.3f}  (worst of all models)")
print(f"\nInflation factor: {knn['CV_PR_AUC']/knn['PR_AUC']:.0f}x")

### 5.2 Why simple additive models win — and the strongest fraud signal
Logistic Regression matches the boosted-tree and stacking ensembles because (i) the anonymized features 
are already pre-engineered, (ii) interactions are weak (all `|r| < 0.1`), and (iii) only 441 training 
fraud cases starve high-capacity models. The single strongest predictor is the **missing-previous-
address flag** — which validates encoding missingness explicitly instead of silently imputing it.

In [ ]:
lr = best_est["LogReg"]
coef = pd.Series(lr.coef_[0], index=feat_names)
top = coef.reindex(coef.abs().sort_values(ascending=False).index).head(12)
print("Top Logistic-Regression coefficients (standardized features):\n")
for f, c in top.items():
    print(f"  {f:35s} {c:+.3f}")

### 5.3 The accuracy mirage
Random Forest posts the **highest accuracy (~98 %)** yet catches only ~20 % of fraud at a 0.5 threshold — 
barely above the 98.9 % floor of a do-nothing classifier. This is exactly why we drive selection by 
PR-AUC and budgeted recall rather than accuracy.

In [ ]:
cm_rf = cms["RandomForest"]
caught = cm_rf[1, 1]; total_fraud = cm_rf[1].sum()
print(f"Random Forest accuracy : {comparison.set_index('Model').loc['RandomForest','Accuracy']:.3f}")
print(f"Trivial 'always legit' : {1 - y_te.mean():.3f}")
print(f"Fraud actually caught  : {caught}/{total_fraud}  ({caught/total_fraud:.0%})")

## 6. Conclusion

**Key findings**
- A properly weighted **Logistic Regression matches gradient boosting and stacking** (ROC-AUC ≈ 0.91, 
  recall ≈ 54 % at a 5 % false-positive budget) on this anonymized, weakly-interacting feature space.
- **Explicit missing-value indicators** carry strong, interpretable signal (missing-prev-address is #1).
- Applying **SMOTE outside the CV loop** produces a dramatic, cautionary validation-leakage artifact.
- Under extreme imbalance, evaluate by **PR-AUC and budgeted recall — never accuracy.**

**Limitations** — 50k subsample (441 training fraud) caps achievable PR-AUC; single train/test split; 
SVM tuned/refit on subsamples; no probability calibration or fairness audit.

**Future work** — scale to the full 1M rows with SMOTE *inside* CV; time-based split on the `month` 
field to test concept drift; calibrate probabilities to an explicit cost matrix; audit group fairness.

---
*Reproducible with `SEED = 42`. Built with scikit-learn; `HistGradientBoostingClassifier` stands in for 
LightGBM/XGBoost.*